In [29]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict
import os

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)


class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str


def generate_joke(state: JokeState):
    prompt = f"Generate a joke on the topic: {state['topic']}"

    response = llm.invoke(prompt).content

    return {
        "joke": response
    }


def generate_explanation(state: JokeState):
    prompt = f"Explain the following joke:\n\n{state['joke']}"

    response = llm.invoke(prompt).content

    return {
        "explanation": response
    }


graph = StateGraph(JokeState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

# ---------- Thread 1 ----------
config1 = {
    "configurable": {
        "thread_id": "1"
    }
}

result1 = workflow.invoke(
    {"topic": "pizza"},
    config=config1
)

print(result1)

# ---------- Thread 2 ----------
config2 = {
    "configurable": {
        "thread_id": "2"
    }
}

result2 = workflow.invoke(
    {"topic": "pasta"},
    config=config2
)

print(result2)

# ---------- Get current state ----------
print(workflow.get_state(config1))

# ---------- State history ----------
history = list(workflow.get_state_history(config1))

print(history)

{'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'A classic play on words. The joke is funny because it uses a pun to create a wordplay. \n\nIn this joke, "crusty" has a double meaning:\n1. A pizza has a crust, which is the outer layer of the pizza. So, "crusty" can refer to the physical crust of the pizza.\n2. "Crusty" can also mean being irritable, grumpy, or having a bad temper. For example, "He\'s been feeling a little crusty all day."\n\nThe joke relies on this dual meaning of "crusty" to create the humor. It sets up the expectation that the pizza is in a bad mood, and then subverts it by using the word "crusty" in a way that references both the pizza\'s physical crust and its emotional state. The punchline is unexpected, yet makes sense in a clever way, which creates the comedic effect.'}
{'topic': 'pasta', 'joke': 'Why was the pasta in a bad mood?\n\nBecause it was feeling a little "drained" and was struggli

In [30]:
list(workflow.get_state_history(config1))


[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'A classic play on words. The joke is funny because it uses a pun to create a wordplay. \n\nIn this joke, "crusty" has a double meaning:\n1. A pizza has a crust, which is the outer layer of the pizza. So, "crusty" can refer to the physical crust of the pizza.\n2. "Crusty" can also mean being irritable, grumpy, or having a bad temper. For example, "He\'s been feeling a little crusty all day."\n\nThe joke relies on this dual meaning of "crusty" to create the humor. It sets up the expectation that the pizza is in a bad mood, and then subverts it by using the word "crusty" in a way that references both the pizza\'s physical crust and its emotional state. The punchline is unexpected, yet makes sense in a clever way, which creates the comedic effect.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f171433

In [31]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [32]:
class CrashState(TypedDict):
    input:str
    step1:str
    step2:str


In [33]:
def step_1(state:CrashState)->CrashState:
    print("Step 1 excuted")
    return {"step1":"done","input":state["input"]}

In [34]:
def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [36]:
builder = StateGraph(CrashState)
builder.add_node("step_1",step_1)
builder.add_node("step_2",step_2)
builder.add_node("step_3",step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1","step_2")
builder.add_edge("step_2","step_3")
builder.add_edge("step_3",END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)